# Training (Spark ML, GCP YARN)
Trains baseline ML models on feature set and evaluates on val/test.
Outputs are saved to HDFS under /user/tiennd.

In [ ]:
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from xgboost.spark import SparkXGBRegressor
from pyspark.ml.evaluation import RegressionEvaluator

BASE_HDFS = "/user/tiennd"
FEATURE_PATH = f"{BASE_HDFS}/feature_engineering/demand_prediction_features_30m"
OUT_BASE_ROOT = f"{BASE_HDFS}/results/sparkml"
MODEL_BASE = f"{BASE_HDFS}/models/sparkml"
TARGET_COL = "pickup_demand_t1"

feature_cols = [
    "hour", "dow", "month", "is_weekend",
    "lag_6", "lag_12", "lag_336",
    "roll_mean_12", "roll_mean_48", "roll_std_48", "cluster_id",
]

spark = (
    SparkSession.builder
    .appName("DemandPredictionTraining_GCP")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", f"hdfs://{BASE_HDFS}/spark-logs")
    .config("spark.executor.instances", "3")
    .config("spark.executor.cores", "3")
    .config("spark.executor.memory", "6g")
    .config("spark.executor.memoryOverhead", "1g")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.memoryOverhead", "1g")
    .config("spark.sql.shuffle.partitions", "96")
    .getOrCreate()
 )
spark.sparkContext.setLogLevel("WARN")

df_all = spark.read.parquet(FEATURE_PATH)
train_df = df_all.filter(F.col("split") == "train").cache()
val_df = df_all.filter(F.col("split") == "val").cache()
test_df = df_all.filter(F.col("split") == "test").cache()
print("Train/Val/Test rows:", train_df.count(), val_df.count(), test_df.count())

In [ ]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
rmse_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")
mae_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
r2_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="r2")

def add_smape(df):
    denom = (F.abs(F.col(TARGET_COL)) + F.abs(F.col("prediction")))
    smape = F.when(denom == 0, F.lit(0.0)).otherwise(200.0 * F.abs(F.col(TARGET_COL) - F.col("prediction")) / denom)
    return df.withColumn("smape", smape)

def evaluate(model_name, pred_df):
    pred_df = pred_df.withColumn("prediction", F.when(F.col("prediction") < 0, 0.0).otherwise(F.col("prediction")))
    rmse = float(rmse_eval.evaluate(pred_df))
    mae = float(mae_eval.evaluate(pred_df))
    r2 = float(r2_eval.evaluate(pred_df))
    mape = float(pred_df.agg(F.avg(F.abs(F.col(TARGET_COL) - F.col("prediction")) / F.col(TARGET_COL)) * 100.0).first()[0])
    smape = float(add_smape(pred_df).agg(F.avg(F.col("smape"))).first()[0])
    return {"model": model_name, "RMSE": rmse, "MAE": mae, "MAPE": mape, "sMAPE": smape, "R2": r2}

models = {
    "linear_regression": LinearRegression(featuresCol="features", labelCol=TARGET_COL, predictionCol="prediction", maxIter=50),
    "random_forest": RandomForestRegressor(featuresCol="features", labelCol=TARGET_COL, predictionCol="prediction", numTrees=50, maxDepth=8, seed=42),
    "xgboost": SparkXGBRegressor(features_col="features", label_col=TARGET_COL, prediction_col="prediction", n_estimators=80, max_depth=6, learning_rate=0.05, subsample=0.8, num_workers=3),
}

metrics_rows = []
pred_union = None
fitted_models = {}

for name, model in models.items():
    pipeline = Pipeline(stages=[assembler, model])
    fitted = pipeline.fit(train_df)
    fitted_models[name] = fitted
    val_pred = fitted.transform(val_df)
    test_pred = fitted.transform(test_df)
    val_metrics = evaluate(name + "_val", val_pred)
    test_metrics = evaluate(name + "_test", test_pred)
    metrics_rows.extend([val_metrics, test_metrics])
    slim_pred = test_pred.select(F.lit(name).alias("model"), "PULocationID", "pickup_bin_30m", TARGET_COL, "prediction")
    pred_union = slim_pred if pred_union is None else pred_union.unionByName(slim_pred)
    print("Finished model:", name)

In [ ]:
metrics_pdf = pd.DataFrame(metrics_rows)
display(metrics_pdf)

val_rows = metrics_pdf[metrics_pdf["model"].str.endswith("_val")].copy()
best_val = val_rows.sort_values("sMAPE").iloc[0]["model"].replace("_val", "")
print("Best model by val sMAPE:", best_val)

run_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
OUT_BASE = f"{OUT_BASE_ROOT}/run_{run_id}"
MODEL_PATH = f"{MODEL_BASE}/run_{run_id}/{best_val}"

metrics_sdf = spark.createDataFrame(metrics_rows)
metrics_sdf.write.mode("overwrite").parquet(f"{OUT_BASE}/metrics")
pred_union.write.mode("overwrite").partitionBy("model").parquet(f"{OUT_BASE}/predictions")
fitted_models[best_val].write().overwrite().save(MODEL_PATH)

print("Saved:")
print("-", f"{OUT_BASE}/metrics")
print("-", f"{OUT_BASE}/predictions")
print("-", MODEL_PATH)

In [ ]:
spark.catalog.clearCache()
spark.stop()